# An OUTPUT PARSER converts an LLM’s text response into a usable Python structure such as:
**-Pydantic object**<br>
**-Python dictionary**<br>
**-Selected route**<br>
**-Validated Guardrails output**<br>

<div align="center">
    <img src="images/outputparser.png" width="500" height="700">
</div>

| Parser                             | Main purpose                     | Structure defined using       | Returns              |                
| ---------------------------------- | -------------------------------- | ----------------------------- | -------------------- | 
| `PydanticOutputParser`             | Generate typed structured data   | Pydantic model                | Pydantic object      |
| `SelectionOutputParser`            | Select options with reasons      | Built-in choice structure     | Selection objects    | 
| LangChain `StructuredOutputParser` | Generate general structured data | `ResponseSchema`              | Dictionary           |        
| Guardrails                         | Validate and control output      | Pydantic model and validators | Validated dictionary | 
| Custom `BaseOutputParser`          | Create your own parsing rules    | Custom Python code            | Any chosen type      |  


In [1]:
import logging
logging.getLogger("opentelemetry").setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry.exporter.otlp.proto.http.trace_exporter").setLevel(logging.CRITICAL)

In [2]:
from llama_index.llms.openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
llm = OpenAI(model="gpt-4o-mini", temperature=0)

In [3]:
from pydantic import BaseModel, Field

class CourseInfo(BaseModel):
    topic: str = Field(description="Course topic")
    level: str = Field(description="Difficulty level")
    description: str = Field(description="Short description")

<h1 style="color: #16A34A;">1. PYDANTIC OUTPUT PARSER</h1>

# PydanticOutputParser tells the LLM to produce output matching a Pydantic model.

In [5]:
from llama_index.core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(output_cls=CourseInfo)

prompt = """
Explain Python programming for a beginner.

{format_instructions}
"""
formatted_prompt = prompt.format(format_instructions=parser.format(""))
response = llm.complete(formatted_prompt)
result = parser.parse(str(response))
result_dict = result.model_dump()
print(json.dumps(result.model_dump(), indent=4))

{
    "topic": "Python Programming",
    "level": "Beginner",
    "description": "This course introduces the fundamentals of Python programming, covering basic syntax, data types, control structures, functions, and simple data manipulation. Ideal for those new to coding."
}


<h1 style="color: #16A34A;">2. SELECTION OUTPUT PARSER</h1>

# It is mainly designed for selection/routing decisions, where the LLM needs to select one or more choices and give a reason. LlamaIndex's implementation parses JSON into structured Answer objects.

In [7]:
from llama_index.core.output_parsers import SelectionOutputParser

parser = SelectionOutputParser()
prompt = """
Select the correct option.
Options:
1. Vector Database
2. SQL Database
3. Web Search

Question:
Find information from a PDF document.
Return the selected option and the reason.

""" + parser.format("")

response = llm.complete(prompt)
result = parser.parse(str(response))
answer = result.parsed_output[0]
print(f"'choice': {answer.choice},")
print(f"'reason': '{answer.reason}'")

'choice': 1,
'reason': 'A Vector Database is designed to handle unstructured data and can efficiently search through embeddings, which is useful for retrieving information from documents like PDFs.'


<h1 style="color: #16A34A;">3. LANGCHAIN  OUTPUT PARSER</h1>

# This parser allows you to use a LangChain output parser inside LlamaIndex.

In [10]:
from langchain.output_parsers import StructuredOutputParser, ResponseSchema

# Define the expected output fields
response_schemas = [
    ResponseSchema(name="topic", description="The main topic"),
    ResponseSchema(name="level", description="The difficulty level"),
    ResponseSchema(name="description", description="Short description")
]
# Create LangChain output parser
parser = StructuredOutputParser.from_response_schemas(response_schemas)

# Get formatting instructions
format_instructions = parser.get_format_instructions()

# Create prompt
prompt = f"""
Explain Python programming for a beginner.

{format_instructions}
"""
# Generate response using LlamaIndex
response = llm.complete(prompt)

# Convert LlamaIndex response to string
raw_output = str(response)

# Parse using LangChain parser
result = parser.parse(raw_output)
print(json.dumps(result, indent=4))

{
    "topic": "Python Programming",
    "level": "Beginner",
    "description": "Python is a high-level, interpreted programming language known for its readability and simplicity. It is widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Beginners can easily learn Python due to its straightforward syntax and large supportive community."
}


<h1 style="color: #16A34A;">4. GUARDRAILS OUTPUT PARSER</h1>

# GuardrailsOutputParser is useful when you want schema-based validation and correction of LLM output using Guardrails.

In [11]:
from guardrails import Guard

# Create Guardrails guard
guard = Guard.for_pydantic(CourseInfo)

# Create prompt
prompt = """
Explain Python programming for a beginner.

Return the answer as JSON with exactly these fields:
topic
level
description
"""
# Generate response
response = llm.complete(prompt)
raw_output = str(response)
validation_result = guard.parse(llm_output=raw_output)

if validation_result.validation_passed:
    print("Validation Status: PASSED")
    print("\nValidated Output:", validation_result.validated_output)
else:
    print("Validation Status: FAILED")
    print("Validation Error:", validation_result.error)
print(json.dumps(result, indent=4))

Validation Status: PASSED

Validated Output: {'topic': 'Python Programming', 'level': 'Beginner', 'description': "Python is a high-level, interpreted programming language known for its readability and simplicity. It is widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Beginners can start by learning basic concepts such as variables, data types, control structures (if statements, loops), functions, and modules. Python's extensive libraries and frameworks make it a versatile choice for various applications. The syntax is designed to be clear and straightforward, making it an excellent language for newcomers to programming."}
{
    "topic": "Python Programming",
    "level": "Beginner",
    "description": "Python is a high-level, interpreted programming language known for its readability and simplicity. It is widely used for web development, data analysis, artificial intelligence, scientific computing, and more. Beginners can easily l

C:\Users\User\anaconda3\envs\ai\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


<h1 style="color: #16A34A;">5. BASE OUTPUT PARSER</h1>

# BaseOutputParser is the base abstraction from which you can create your own output parser.

In [12]:
from llama_index.core.types import BaseOutputParser

class SimpleCourseParser(BaseOutputParser):
    def parse(self, output: str):
        parts = output.split("|")
        return {
            "topic": parts[0].strip(),
            "level": parts[1].strip(),
            "description": parts[2].strip()
        }
    def format(self, prompt_template: str):
        return prompt_template + """
Return the answer in this exact format:
topic | level | description
"""
parser = SimpleCourseParser()
prompt = parser.format("Explain Python programming for a beginner.")
response = llm.complete(prompt)
result = parser.parse(str(response))
print(result)

{'topic': 'variables', 'level': 'beginner', 'description': 'Variables are used to store data values. In Python, you can create a variable by simply assigning a value to a name using the equals sign (`=`). For example, `x = 5` creates a variable `x` that holds the integer value 5.\n\ndata types'}
